![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

https://www.quantconnect.com/datasets/quantconnect-cash-indices

In [1]:
from QuantConnect import *
from QuantConnect.Research import QuantBook
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

qb = QuantBook()

start = pd.Timestamp(2022, 5, 1, tz="UTC") 
end = pd.Timestamp(2023, 5, 1, tz="UTC")    


# Tier 1 indices for BTC prediction
tier1_indices = ['VIX', 'SPX', 'NDX', 'DXY', 'GVZ']

# Dictionary to store all index data
index_data = {}

print("Fetching Tier 1 Indices Data...")
print("=" * 50)

# Modify the data fetching code to use fill_forward more aggressively
for ticker in tier1_indices:
    try:
        print(f"\nFetching {ticker}...")
        
        symbol = qb.add_index(ticker, Resolution.MINUTE).symbol
        
        # IMPORTANT: fill_forward=True carries last value during non-trading hours
        history = qb.history(symbol, start, end, Resolution.MINUTE, fill_forward=True)
        
        if history.empty:
            print(f"  ✗ No data for {ticker}")
            continue
        
        # Reset and resample
        history = history.reset_index()
        history['time_5min'] = history['time'].dt.floor('5min')
        
        # Use 'last' for all aggregations to preserve forward-filled values
        index_5min = history.groupby('time_5min').agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last'
        })
        
        index_data[ticker] = index_5min
        print(f"  ✓ {len(index_5min)} bars")
        
    except Exception as e:
        print(f"  ✗ Error fetching {ticker}: {str(e)}")

print("\n" + "=" * 50)
print(f"Successfully fetched {len(index_data)} out of {len(tier1_indices)} indices")

# Display summary
if index_data:
    print("\nData Summary:")
    for ticker, data in index_data.items():
        print(f"{ticker}: {len(data)} bars, Close range: {data['close'].min():.2f} - {data['close'].max():.2f}")

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

def add_index_features(df, ticker):
    """
    Stationary features for indices as BTC predictors.
    Focus: Risk sentiment, volatility regime, equity momentum.
    All features prefixed with index_{ticker}_. No raw OHLC in output.
    """
    features = df.copy()
    
    # ============================================================
    # PRICE MOMENTUM (Returns - stationary)
    # ============================================================
    features['ret_1'] = features['close'].pct_change(1)      # 5-min
    features['ret_3'] = features['close'].pct_change(3)      # 15-min
    features['ret_12'] = features['close'].pct_change(12)    # 1-hour
    features['ret_48'] = features['close'].pct_change(48)    # 4-hour
    features['ret_288'] = features['close'].pct_change(288)  # 1-day
    
    # Momentum acceleration
    features['ret_accel_12'] = features['ret_12'] - features['ret_12'].shift(12)
    features['ret_accel_48'] = features['ret_48'] - features['ret_48'].shift(48)
    
    # Momentum percentile (strength of move)
    features['ret_pctrank_12'] = features['ret_12'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    features['ret_pctrank_48'] = features['ret_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # VOLATILITY (Realized vol - key for VIX especially)
    # ============================================================
    features['rvol_12'] = features['ret_1'].rolling(12).std()   # 1-hour
    features['rvol_48'] = features['ret_1'].rolling(48).std()   # 4-hour
    features['rvol_288'] = features['ret_1'].rolling(288).std() # 1-day
    
    # Volatility ratios (regime detection)
    features['vol_ratio_12_48'] = features['rvol_12'] / (features['rvol_48'] + 1e-8)
    features['vol_ratio_48_288'] = features['rvol_48'] / (features['rvol_288'] + 1e-8)
    
    # Volatility percentile
    features['vol_pctrank_48'] = features['rvol_48'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # Volatility regime change
    features['vol_regime_change'] = (features['rvol_48'] - features['rvol_48'].shift(48)) / (features['rvol_288'] + 1e-8)
    
    # ============================================================
    # RANGE-BASED FEATURES
    # ============================================================
    # Intrabar range as % of price
    features['range_pct'] = (features['high'] - features['low']) / features['close']
    features['range_pct_ma12'] = features['range_pct'].rolling(12).mean()
    features['range_pct_ma48'] = features['range_pct'].rolling(48).mean()
    
    # Range percentile
    features['range_pctrank_48'] = features['range_pct'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # Parkinson volatility estimator
    log_hl = np.log(features['high'] / features['low'])
    features['parkinson_vol_12'] = np.sqrt((log_hl ** 2).rolling(12).mean() / (4 * np.log(2)))
    features['parkinson_vol_48'] = np.sqrt((log_hl ** 2).rolling(48).mean() / (4 * np.log(2)))
    
    # ============================================================
    # MEAN REVERSION / TREND SIGNALS
    # ============================================================
    # Z-score (deviation from mean)
    sma_48 = features['close'].rolling(48).mean()
    sma_288 = features['close'].rolling(288).mean()
    std_48 = features['close'].rolling(48).std()
    std_288 = features['close'].rolling(288).std()
    
    features['zscore_48'] = (features['close'] - sma_48) / (std_48 + 1e-8)
    features['zscore_288'] = (features['close'] - sma_288) / (std_288 + 1e-8)
    
    # Moving average positioning
    features['ma_cross_48_288'] = (sma_48 - sma_288) / features['close']
    
    # Price percentile within lookback
    features['price_pctrank_48'] = features['close'].rolling(48).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    features['price_pctrank_288'] = features['close'].rolling(288).apply(
        lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
    )
    
    # ============================================================
    # MOMENTUM QUALITY
    # ============================================================
    # Directional consistency
    features['positive_ret_ratio_12'] = features['ret_1'].rolling(12).apply(lambda x: (x > 0).sum() / len(x))
    features['positive_ret_ratio_48'] = features['ret_1'].rolling(48).apply(lambda x: (x > 0).sum() / len(x))
    
    # Return distribution (tail risk indicators)
    features['ret_skew_48'] = features['ret_1'].rolling(48).skew()
    features['ret_skew_288'] = features['ret_1'].rolling(288).skew()
    features['ret_kurt_48'] = features['ret_1'].rolling(48).kurt()
    
    # ============================================================
    # LEVEL-SPECIFIC FEATURES (for VIX)
    # ============================================================
    # VIX level regimes matter for risk sentiment
    if ticker == 'VIX':
        # VIX level percentile (absolute level matters)
        features['level_pctrank_288'] = features['close'].rolling(288).apply(
            lambda x: stats.percentileofscore(x, x.iloc[-1]) / 100.0 if len(x) > 1 else 0.5
        )
        
        # VIX spikes (rapid increases)
        features['spike_1h'] = (features['close'] - features['close'].shift(12)) / (features['close'].shift(12) + 1e-8)
        features['spike_4h'] = (features['close'] - features['close'].shift(48)) / (features['close'].shift(48) + 1e-8)
    
    # ============================================================
    # SEASONALITY
    # ============================================================
    # Return vs same time yesterday (intraday patterns)
    features['ret_vs_1d_ago'] = features['close'].pct_change(288) / (features['ret_288'].rolling(288).std() + 1e-8)
    
    # ============================================================
    # DROP RAW OHLC (keep only derived features)
    # ============================================================
    features = features.drop(['open', 'high', 'low', 'close'], axis=1, errors='ignore')
    
    # ============================================================
    # PREFIX ALL COLUMNS WITH index_{ticker}_
    # ============================================================
    rename_dict = {col: f'index_{ticker.lower()}_{col}' for col in features.columns}
    features = features.rename(columns=rename_dict)
    
    return features


# Apply to all indices
print("="*60)
print("CREATING INDEX FEATURES (STATIONARY ONLY)")
print("="*60)

index_features = {}

for ticker, df in index_data.items():
    print(f"\n{ticker}...", end=" ")
    features = add_index_features(df.copy(), ticker)
    index_features[ticker] = features
    print(f"✓ {features.shape[1]} features, {features.shape[0]} rows")

print("\n" + "="*60)
print(f"Total indices processed: {len(index_features)}")

In [3]:
# Merge all index features together (outer join to keep all timestamps)
print("Merging all index features...")

# Start with first index
tickers = list(index_features.keys())
merged_indices = index_features[tickers[0]].copy()
print(f"  ✓ Starting with {tickers[0]}: {merged_indices.shape[1]} features")

# Merge remaining indices with outer join
for ticker in tickers[1:]:
    merged_indices = merged_indices.join(index_features[ticker], how='outer')
    print(f"  ✓ Merged {ticker}: {index_features[ticker].shape[1]} features")

print(f"\nMerged indices dataset: {merged_indices.shape[0]} rows × {merged_indices.shape[1]} columns")

# Check NaN distribution
total_nans = merged_indices.isna().sum().sum()
total_cells = merged_indices.shape[0] * merged_indices.shape[1]
print(f"Total NaNs: {total_nans:,} / {total_cells:,} ({total_nans/total_cells*100:.1f}%)")

NANS TO MARKET DIFFERENCE PERIOD. KEEP NANS FOR NOW BUT ASSES IN MEGAMERGE

In [4]:
# Save merged indices dataset
merged_indices.to_parquet('indices_all_merged.parquet')
print(f"✓ Saved indices_all_merged.parquet")
print(f"  Shape: {merged_indices.shape}")
print(f"  Date range: {merged_indices.index.min()} to {merged_indices.index.max()}")